In [22]:
import numpy as np
import pandas as pd

### Part I: Aggregate sales data
Load all transaction data, noting down close year and month for downstream model training and tuning

In [23]:
months_2024 = [f"{m:02d}" for m in range(1, 13)]
files_2024 = [f"raw-data/CRMLSSold2024{m}.csv" for m in months_2024]
sold_2024 = pd.concat(pd.read_csv(f) for f in files_2024)

/var/folders/t2/p9112v_n469068__fty8_3nc0000gn/T/ipykernel_23575/66980963.py:3: DtypeWarning: Columns (2,36,39,56,74) have mixed types. Specify dtype option on import or set low_memory=False.
  sold_2024 = pd.concat(pd.read_csv(f) for f in files_2024)


In [24]:
months_2025 = [f"{m:02d}" for m in range(1, 13)]
files_2025 = [f"raw-data/CRMLSSold2025{m}.csv" for m in months_2025]
sold_2025 = pd.concat(pd.read_csv(f) for f in files_2025)

/var/folders/t2/p9112v_n469068__fty8_3nc0000gn/T/ipykernel_23575/2082103420.py:3: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  sold_2025 = pd.concat(pd.read_csv(f) for f in files_2025)


In [25]:
months_2026 = [f"{m:02d}" for m in range(1, 6)]
files_2026 = [f"raw-data/CRMLSSold2026{m}.csv" for m in months_2026]
sold_2026 = pd.concat(pd.read_csv(f) for f in files_2026)

/var/folders/t2/p9112v_n469068__fty8_3nc0000gn/T/ipykernel_23575/1279266054.py:3: DtypeWarning: Columns (4,74) have mixed types. Specify dtype option on import or set low_memory=False.
  sold_2026 = pd.concat(pd.read_csv(f) for f in files_2026)


In [26]:
sold = pd.concat([sold_2024, sold_2025, sold_2026])
sold.shape

(639254, 84)

### Part II: Filter for residential sales

Restrict analysis to single-family residential properties

In [27]:
sold_residential = sold[sold['PropertyType'] == 'Residential']
sold_sfr = sold_residential[sold_residential['PropertySubType'] == 'SingleFamilyResidence']
sold_sfr.shape

(321846, 84)

### Part III: Handle missing values

Profile missing values in key columns (e.g., close price, living area, bedrooms, bathrooms, lot size)

In [28]:
sold_sfr = sold_sfr.replace(r'^\s*$', np.nan, regex=True)
null_counts = sold_sfr.isna().sum()
null_percs =  sold_sfr.isna().mean() * 100
null_table = pd.DataFrame({
    'null count': null_counts,
    'null percentage': null_percs
})

/var/folders/t2/p9112v_n469068__fty8_3nc0000gn/T/ipykernel_23575/2925654690.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  sold_sfr = sold_sfr.replace(r'^\s*$', np.nan, regex=True)


In [29]:
null_table.loc[[
    'ClosePrice', 'LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'DaysOnMarket', 'YearBuilt',
    'LotSizeArea', 'LotSizeAcres', 'LotSizeSquareFeet',
    'City', 'CountyOrParish', 'PostalCode', 'Latitude', 'Longitude',
    'CloseDate', 'ListingContractDate', 'PurchaseContractDate'
]]

,null count,null percentage
ClosePrice,2,0.000621
LivingArea,173,0.053752
BedroomsTotal,0,0.000000
BathroomsTotalInteger,50,0.015535
DaysOnMarket,0,0.000000
YearBuilt,232,0.072084
LotSizeArea,5516,1.713863
LotSizeAcres,5568,1.730020
LotSizeSquareFeet,5540,1.721320
City,246,0.076434


Drop records with missing close price, impute missing living area, bathrooms, and lot size with respective county medians, and flag records with missing city, postal code, latitude, or longitude

In [30]:
# drop records with missing close price (target variable)
sold_sfr = sold_sfr.dropna(subset='ClosePrice')

In [31]:
# impute records with missing configuration details
cols = ['LivingArea', 'BathroomsTotalInteger', 'LotSizeArea', 'LotSizeAcres', 'LotSizeSquareFeet']
sold_sfr[cols] = sold_sfr[cols].apply(pd.to_numeric)

for col in cols:
    sold_sfr[col] = sold_sfr[col].fillna(
        sold_sfr.groupby('CountyOrParish')[col].transform('median')
    )

In [32]:
# flag records with missing location details
sold_sfr['missing_city'] = sold_sfr['City'].isna()
sold_sfr['missing_postalcode'] = sold_sfr['PostalCode'].isna()
sold_sfr['missing_lat'] = sold_sfr['Latitude'].isna()
sold_sfr['missing_lon'] = sold_sfr['Longitude'].isna()

### Part IV: Create train-test split

Use records from the most recent month for testing and records in the preceding months for training

In [33]:
# hyperparameter to adjust
train_months = 12

close_month = pd.to_datetime(sold_sfr['CloseDate']).dt.to_period('M')
latest_month = close_month.max()

train_start = latest_month - train_months
train_set = sold_sfr[(close_month >= train_start) & (close_month < latest_month)]
test_set = sold_sfr[close_month == latest_month]

In [34]:
train_set.to_csv('clean-data/sold_train.csv', index=False)
test_set.to_csv('clean-data/sold_test.csv', index=False)